# Initial Libraries

In [14]:
from __future__ import annotations
from pathlib import Path
from dataclasses import dataclass, field
from collections import defaultdict
from copy import deepcopy
from typing import Dict, List, Optional, Tuple, Iterable
import math
import re
from tqdm import tqdm
import pandas as pd

# Heuristic $\mathcal{H}$

In [2]:
EPS = 1e-9


@dataclass(frozen=True)
class Hole:
    start: float
    end: float

    def __post_init__(self) -> None:
        if self.end <= self.start:
            raise ValueError(f"Invalid hole [{self.start}, {self.end}].")


@dataclass
class Operation:
    machine: str
    duration: float
    alpha: float = 0.0          # 0.0 = fully resumable, 1.0 = full restart after interruption
    crossable: bool = True      # if False, interruption implies full restart
    name: str = ""

    def __post_init__(self) -> None:
        if self.duration <= 0:
            raise ValueError(f"Operation duration must be > 0, got {self.duration}.")
        if not (0.0 <= self.alpha <= 1.0):
            raise ValueError(f"alpha must be in [0, 1], got {self.alpha}.")


@dataclass
class Job:
    job_id: str
    operations: List[Operation]

    def __post_init__(self) -> None:
        if not self.operations:
            raise ValueError(f"Job {self.job_id} must contain at least one operation.")


@dataclass
class Segment:
    job_id: str
    op_index: int       # 0-based
    machine: str
    start: float
    end: float


class FlowScheduler:
    """
    Flow-based multidimensional heuristic with:
      - event-driven simulation,
      - machine holes,
      - portfolio conflict resolution,
      - short rollout candidate evaluation.

    Main assumptions:
      - one machine per operation,
      - no voluntary preemption,
      - interruption can happen only because of a machine hole,
      - alpha is interpreted as 'fraction of current uninterrupted segment lost'
        when interrupted by a hole.
    """

    def __init__(
        self,
        jobs: List[Job],
        holes_by_machine: Dict[str, List[Hole]],
        rules: Optional[List[str]] = None,
        rollout_events: int = 2,
        rollout_base_rule: str = "eft",
    ) -> None:
        if not jobs:
            raise ValueError("At least one job is required.")

        self.jobs: Dict[str, Job] = {job.job_id: deepcopy(job) for job in jobs}
        if len(self.jobs) != len(jobs):
            raise ValueError("Job IDs must be unique.")

        self.machine_holes: Dict[str, List[Hole]] = {
            m: sorted(deepcopy(hs), key=lambda h: (h.start, h.end))
            for m, hs in holes_by_machine.items()
        }

        self.machines: List[str] = sorted(
            {
                op.machine
                for job in jobs
                for op in job.operations
            }.union(self.machine_holes.keys())
        )

        self.rules = rules or ["eft", "spt","lpt", "mwr", "least_blocking","most_operations_remaining", "fcfs"]
        self.rollout_events = max(0, rollout_events)
        self.rollout_base_rule = rollout_base_rule

        self.time: float = 0.0

        # Per-job state
        self.op_index: Dict[str, int] = {jid: 0 for jid in self.jobs}
        self.remaining: Dict[str, float] = {
            jid: self.jobs[jid].operations[0].duration for jid in self.jobs
        }
        self.segment_progress: Dict[str, float] = {jid: 0.0 for jid in self.jobs}
        self.job_running: Dict[str, bool] = {jid: False for jid in self.jobs}
        self.open_segment_start: Dict[str, Optional[float]] = {jid: None for jid in self.jobs}

        # Per-machine state
        self.machine_running: Dict[str, Optional[str]] = {m: None for m in self.machines}

        # Outputs
        self.segments: List[Segment] = []
        self.completion_times: Dict[Tuple[str, int], float] = {}

        self.request_time: Dict[str, Optional[float]] = {
            jid: 0.0 if not self.is_finished(jid) else None
            for jid in self.jobs
        }

    # ------------------------------------------------------------------
    # Basic state helpers
    # ------------------------------------------------------------------

    def current_operation(self, job_id: str) -> Optional[Operation]:
        idx = self.op_index[job_id]
        ops = self.jobs[job_id].operations
        return ops[idx] if idx < len(ops) else None

    def is_finished(self, job_id: str) -> bool:
        return self.op_index[job_id] >= len(self.jobs[job_id].operations)

    def unfinished_jobs(self) -> List[str]:
        return [jid for jid in self.jobs if not self.is_finished(jid)]

    def current_machine(self, job_id: str) -> Optional[str]:
        op = self.current_operation(job_id)
        return None if op is None else op.machine

    def _loss_fraction(self, op: Operation) -> float:
        return 1.0 if not op.crossable else op.alpha

    def _machine_holes(self, machine: str) -> List[Hole]:
        return self.machine_holes.get(machine, [])

    # ------------------------------------------------------------------
    # Availability calendar
    # ------------------------------------------------------------------

    def machine_available(self, machine: str, t: float) -> bool:
        for hole in self._machine_holes(machine):
            if hole.start - EPS <= t < hole.end - EPS:
                return False
        return True

    def next_boundary(self, machine: str, t: float) -> Optional[float]:
        """
        Next availability change strictly after or at t if currently in a hole.
        - If machine is available at t: returns next hole start > t.
        - If machine is unavailable at t: returns current hole end.
        """
        for hole in self._machine_holes(machine):
            if hole.start - EPS <= t < hole.end - EPS:
                return hole.end
            if t < hole.start - EPS:
                return hole.start
        return None

    # ------------------------------------------------------------------
    # Effective completion-time operator Gamma_i(t, q_i)
    # ------------------------------------------------------------------

    def earliest_completion_time(
        self,
        job_id: str,
        start_time: Optional[float] = None,
        remaining: Optional[float] = None,
        segment_progress: Optional[float] = None,
    ) -> float:
        """
        Earliest completion time if the current operation of job_id is allocated
        its machine from start_time onward.

        The machine calendar is respected exactly.
        alpha is interpreted as the fraction of progress lost when a hole
        interrupts the current uninterrupted segment.
        """
        if self.is_finished(job_id):
            return self.time if start_time is None else start_time

        t = self.time if start_time is None else start_time
        q = self.remaining[job_id] if remaining is None else remaining
        seg = self.segment_progress[job_id] if segment_progress is None else segment_progress

        op = self.current_operation(job_id)
        if op is None:
            return t

        loss_fraction = self._loss_fraction(op)
        holes = self._machine_holes(op.machine)

        while q > EPS:
            # If inside a hole, wait until the hole ends.
            inside_hole = False
            for hole in holes:
                if hole.start - EPS <= t < hole.end - EPS:
                    t = hole.end
                    seg = 0.0
                    inside_hole = True
                    break
            if inside_hole:
                continue

            next_hole = None
            for hole in holes:
                if hole.start > t + EPS:
                    next_hole = hole
                    break

            if next_hole is None:
                return t + q

            free_window = next_hole.start - t
            if q <= free_window + EPS:
                return t + q

            # Work until the next hole.
            q -= free_window
            seg += free_window

            # Interruption at hole start: lose a fraction of progress in the
            # current uninterrupted segment.
            q += loss_fraction * seg

            # Wait through the hole, then restart/resume.
            t = next_hole.end
            seg = 0.0

        return t

    def effective_duration(self, job_id: str) -> float:
        return self.earliest_completion_time(job_id) - self.time

    # ------------------------------------------------------------------
    # Progress-space quantities
    # ------------------------------------------------------------------

    def direction_vector(self) -> Dict[str, float]:
        running = [jid for jid, flag in self.job_running.items() if flag]
        if not running:
            return {}

        inv_sum = 0.0
        tau: Dict[str, float] = {}
        for jid in running:
            tau_j = self.effective_duration(jid)
            tau[jid] = max(tau_j, EPS)
            inv_sum += 1.0 / tau[jid]

        return {jid: (1.0 / tau[jid]) / inv_sum for jid in running}

    # ------------------------------------------------------------------
    # Request sets and dispatching rules
    # ------------------------------------------------------------------

    def waiting_requesters_by_machine(self) -> Dict[str, List[str]]:
        req: Dict[str, List[str]] = defaultdict(list)
        for jid in self.jobs:
            if self.is_finished(jid) or self.job_running[jid]:
                continue
            op = self.current_operation(jid)
            if op is None:
                continue
            req[op.machine].append(jid)
        return req

    def total_remaining_work(self, job_id: str) -> float:
        if self.is_finished(job_id):
            return 0.0
        idx = self.op_index[job_id]
        ops = self.jobs[job_id].operations
        total = self.remaining[job_id]
        for k in range(idx + 1, len(ops)):
            total += ops[k].duration
        return total

    def total_remaining_operations(self, job_id: str) -> int:
        """
        Number of operations still not completed in the job,
        including the current one if the job is unfinished.
        """
        if self.is_finished(job_id):
            return 0
        return len(self.jobs[job_id].operations) - self.op_index[job_id]

    def blocking_score(self, job_id: str) -> int:
        """
        Crude proxy for downstream blocking.
        Lower is better.
        """
        if self.is_finished(job_id):
            return 0

        idx = self.op_index[job_id]
        ops = self.jobs[job_id].operations
        next_machine = ops[idx + 1].machine if idx + 1 < len(ops) else None
        if next_machine is None:
            return 0

        count = 0
        for other in self.unfinished_jobs():
            if other == job_id:
                continue
            op = self.current_operation(other)
            if op is None:
                continue
            if op.machine == next_machine:
                count += 1
            else:
                oidx = self.op_index[other]
                oops = self.jobs[other].operations
                if oidx + 1 < len(oops) and oops[oidx + 1].machine == next_machine:
                    count += 1
        return count

    def propose_winner(self, rule: str, contenders: List[str], machine: str) -> str:
        if not contenders:
            raise ValueError("Empty contender set.")

        if rule in {"eft", "machine_release"}:
            return min(
                contenders,
                key=lambda jid: (self.earliest_completion_time(jid), jid),
            )

        if rule == "spt":
            return min(
                contenders,
                key=lambda jid: (self.remaining[jid], jid),
            )

        if rule == "lpt":
          return max(
              contenders,
              key=lambda jid: (self.remaining[jid], jid),
          )

        if rule == "mwr":
            return min(
                contenders,
                key=lambda jid: (-self.total_remaining_work(jid), jid),
            )

        if rule == "least_blocking":
            return min(
                contenders,
                key=lambda jid: (self.blocking_score(jid), self.earliest_completion_time(jid), jid),
            )

        if rule == "most_operations_remaining":
            return min(
                contenders,
                key=lambda jid: (-self.total_remaining_operations(jid), jid),
            )

        if rule == "fcfs":
            return min(
                contenders,
                key=lambda jid: (
                    float("inf") if self.request_time[jid] is None else self.request_time[jid],
                    jid,
                ),
            )

        raise ValueError(f"Unknown dispatching rule: {rule}")

    def lower_bound_remaining(self) -> float:
        """
        Simple lower bound on remaining makespan from current time.
        """
        unfinished = self.unfinished_jobs()
        if not unfinished:
            return 0.0

        job_lb = max(self.total_remaining_work(jid) for jid in unfinished)

        machine_loads: Dict[str, float] = defaultdict(float)
        for jid in unfinished:
            idx = self.op_index[jid]
            ops = self.jobs[jid].operations
            machine_loads[ops[idx].machine] += self.remaining[jid]
            for k in range(idx + 1, len(ops)):
                machine_loads[ops[k].machine] += ops[k].duration

        machine_lb = max(machine_loads.values()) if machine_loads else 0.0
        return max(job_lb, machine_lb)

    def evaluate_candidate(self, machine: str, candidate: str) -> Tuple[float, float, float, str]:
        """
        Candidate score = (projected time + lower bound, immediate finish, projected time, job_id)
        Lower is better.
        """
        immediate_finish = self.earliest_completion_time(candidate)

        if self.rollout_events <= 0:
            return (immediate_finish, immediate_finish, self.time, candidate)

        sim = deepcopy(self)
        forced = {machine: candidate}

        # First event: enforce the candidate on the contested machine.
        sim.run(
            max_events=self.rollout_events,
            use_rollout=False,
            forced_first_choices=forced,
            default_rule=self.rollout_base_rule,
        )

        projected = sim.time + sim.lower_bound_remaining()
        return (projected, immediate_finish, sim.time, candidate)

    def select_winner(self, machine: str, contenders: List[str]) -> str:
        proposals = {
            self.propose_winner(rule, contenders, machine)
            for rule in self.rules
        }

        if len(proposals) == 1:
            return next(iter(proposals))

        scored = [self.evaluate_candidate(machine, cand) for cand in proposals]
        scored.sort()
        return scored[0][3]

    # ------------------------------------------------------------------
    # Segment management
    # ------------------------------------------------------------------

    def _open_segment(self, job_id: str) -> None:
        if self.open_segment_start[job_id] is None:
            self.open_segment_start[job_id] = self.time

    def _close_segment(self, job_id: str) -> None:
        start = self.open_segment_start[job_id]
        if start is None:
            return
        if self.time > start + EPS:
            op = self.current_operation(job_id)
            # During completion, current_operation still points to the finished op.
            if op is None:
                raise RuntimeError("Cannot close segment without current operation.")
            self.segments.append(
                Segment(
                    job_id=job_id,
                    op_index=self.op_index[job_id],
                    machine=op.machine,
                    start=start,
                    end=self.time,
                )
            )
        self.open_segment_start[job_id] = None

    # ------------------------------------------------------------------
    # Machine allocation and event handling
    # ------------------------------------------------------------------

    def _start_job(self, job_id: str) -> None:
        op = self.current_operation(job_id)
        if op is None:
            return
        machine = op.machine

        if self.job_running[job_id]:
            return
        if self.machine_running[machine] is not None:
            raise RuntimeError(f"Machine {machine} is already occupied.")
        if not self.machine_available(machine, self.time):
            raise RuntimeError(f"Machine {machine} is unavailable at time {self.time}.")

        self.job_running[job_id] = True
        self.machine_running[machine] = job_id
        self._open_segment(job_id)

    def _interrupt_job(self, job_id: str) -> None:
        op = self.current_operation(job_id)
        if op is None:
            return
        machine = op.machine

        self._close_segment(job_id)

        loss_fraction = self._loss_fraction(op)
        if loss_fraction > EPS:
            self.remaining[job_id] += loss_fraction * self.segment_progress[job_id]

        self.segment_progress[job_id] = 0.0
        self.job_running[job_id] = False
        self.machine_running[machine] = None

    def _complete_operation(self, job_id: str) -> None:
        op = self.current_operation(job_id)
        if op is None:
            return
        machine = op.machine
        finished_index = self.op_index[job_id]

        self._close_segment(job_id)

        self.job_running[job_id] = False
        self.machine_running[machine] = None
        self.segment_progress[job_id] = 0.0
        self.remaining[job_id] = 0.0
        self.completion_times[(job_id, finished_index)] = self.time

        self.op_index[job_id] += 1
        if not self.is_finished(job_id):
            next_op = self.current_operation(job_id)
            assert next_op is not None
            self.remaining[job_id] = next_op.duration
            self.request_time[job_id] = self.time
        else:
            self.request_time[job_id] = None

    def allocate_free_machines(
        self,
        use_rollout: bool = True,
        forced_choices: Optional[Dict[str, str]] = None,
        default_rule: str = "eft",
    ) -> None:
        forced_choices = forced_choices or {}
        requests = self.waiting_requesters_by_machine()

        for machine in sorted(requests):
            contenders = requests[machine]
            if not contenders:
                continue
            if self.machine_running[machine] is not None:
                continue
            if not self.machine_available(machine, self.time):
                continue

            forced = forced_choices.get(machine)
            if forced is not None and forced in contenders:
                winner = forced
            elif len(contenders) == 1:
                winner = contenders[0]
            elif use_rollout:
                winner = self.select_winner(machine, contenders)
            else:
                winner = self.propose_winner(default_rule, contenders, machine)

            self._start_job(winner)

    def next_event_time(self) -> Optional[float]:
        candidates: List[float] = []

        # Completion events for currently running jobs.
        for machine, jid in self.machine_running.items():
            if jid is None:
                continue
            candidates.append(self.earliest_completion_time(jid))

        # Availability changes on machines that matter now:
        # busy machines OR machines with waiting requesters.
        waiting = self.waiting_requesters_by_machine()
        relevant_machines = {
            m for m, jid in self.machine_running.items() if jid is not None
        }.union(
            m for m, reqs in waiting.items() if reqs
        )

        for machine in relevant_machines:
            boundary = self.next_boundary(machine, self.time)
            if boundary is not None and boundary > self.time + EPS:
                candidates.append(boundary)

        if not candidates:
            return None

        t_next = min(candidates)
        return t_next if t_next > self.time + EPS else None

    def step(
        self,
        use_rollout: bool = True,
        forced_choices: Optional[Dict[str, str]] = None,
        default_rule: str = "eft",
    ) -> bool:
        if not self.unfinished_jobs():
            return False

        # Allocate any free/available machines first.
        self.allocate_free_machines(
            use_rollout=use_rollout,
            forced_choices=forced_choices,
            default_rule=default_rule,
        )

        t_next = self.next_event_time()
        if t_next is None:
            if not self.unfinished_jobs():
                return False
            waiting = self.waiting_requesters_by_machine()
            if waiting:
                raise RuntimeError(
                    f"Deadlock-like state at time {self.time}: waiting jobs exist but no future event was found."
                )
            return False

        delta = t_next - self.time
        if delta <= EPS:
            raise RuntimeError(f"Non-positive event step encountered: delta={delta}")

        # Advance all running jobs up to the next event.
        running_jobs = [jid for jid, flag in self.job_running.items() if flag]
        for jid in running_jobs:
            self.remaining[jid] -= delta
            self.segment_progress[jid] += delta
            if self.remaining[jid] < EPS:
                self.remaining[jid] = 0.0

        self.time = t_next

        # 1) Complete operations that finish exactly at this event.
        completed_now = [jid for jid in running_jobs if self.remaining[jid] <= EPS]
        for jid in completed_now:
            self._complete_operation(jid)

        # 2) Interrupt jobs hit by a hole boundary at this same event.
        #    If a completion and a hole start coincide, completion takes precedence.
        still_running = [jid for jid, flag in self.job_running.items() if flag]
        for jid in still_running:
            machine = self.current_machine(jid)
            if machine is None:
                continue
            if not self.machine_available(machine, self.time):
                self._interrupt_job(jid)

        return True

    def run(
        self,
        max_events: Optional[int] = None,
        use_rollout: bool = True,
        forced_first_choices: Optional[Dict[str, str]] = None,
        default_rule: str = "eft",
    ) -> Dict[str, object]:
        events = 0
        first = True

        while self.unfinished_jobs():
            forced = forced_first_choices if first else None
            progressed = self.step(
                use_rollout=use_rollout,
                forced_choices=forced,
                default_rule=default_rule,
            )
            if not progressed:
                break

            events += 1
            first = False

            if max_events is not None and events >= max_events:
                break

        return self.summary()

    def summary(self) -> Dict[str, object]:
        makespan = 0.0
        if self.completion_times:
            makespan = max(self.completion_times.values())

        return {
            "time": self.time,
            "makespan": makespan,
            "direction_vector": self.direction_vector(),
            "completion_times": dict(sorted(self.completion_times.items())),
            "segments": list(self.segments),
        }



## Testing Heuristic

In [3]:
jobs = [
      Job(
          job_id="J1",
          operations=[
              Operation(machine="M1", duration=4, alpha=0.0, crossable=True, name="O11"),
              Operation(machine="M2", duration=3, alpha=0.0, crossable=True, name="O12"),
          ],
      ),
      Job(
          job_id="J2",
          operations=[
              Operation(machine="M1", duration=2, alpha=0.0, crossable=True, name="O21"),
              Operation(machine="M2", duration=4, alpha=0.0, crossable=True, name="O22"),
          ],
      ),
      Job(
          job_id="J3",
          operations=[
              Operation(machine="M2", duration=2, alpha=0.0, crossable=True, name="O31"),
              Operation(machine="M1", duration=3, alpha=0.0, crossable=True, name="O32"),
          ],
      ),
  ]

holes = {
    "M1": [Hole(3, 5)],
    "M2": [],
}

In [4]:
scheduler = FlowScheduler(
      jobs=jobs,
      holes_by_machine=holes,
      rules=["eft", "spt","lpt", "mwr", "least_blocking","most_operations_remaining","fcfs"],
      rollout_events=2,
      rollout_base_rule="eft",
  )

In [5]:
result = scheduler.run()

In [6]:
print(f"Makespan: {result['makespan']:.2f}")
print("\nCompletion times:")
for (job_id, op_idx), ctime in result["completion_times"].items():
    print(f"  {job_id}, op {op_idx + 1}: {ctime:.2f}")

print("\nExecuted segments:")
for seg in result["segments"]:
    print(
        f"  {seg.job_id} - op {seg.op_index + 1} on {seg.machine}: "
        f"[{seg.start:.2f}, {seg.end:.2f}]"
    )

Makespan: 11.00

Completion times:
  J1, op 1: 8.00
  J1, op 2: 11.00
  J2, op 1: 2.00
  J2, op 2: 6.00
  J3, op 1: 2.00
  J3, op 2: 11.00

Executed segments:
  J2 - op 1 on M1: [0.00, 2.00]
  J3 - op 1 on M2: [0.00, 2.00]
  J1 - op 1 on M1: [2.00, 3.00]
  J2 - op 2 on M2: [2.00, 6.00]
  J1 - op 1 on M1: [5.00, 8.00]
  J1 - op 2 on M2: [8.00, 11.00]
  J3 - op 2 on M1: [8.00, 11.00]


# Instance Reader

In [7]:
@dataclass
class JSPMUInstance:
    n_jobs: int
    n_machines: int
    jobs: List[Job]
    holes: Dict[str, List[Hole]]

In [8]:
class JSPMUReader:
    """
    Reader for extended ABZ-style .jspmu files.

    Expected format:
        <n_jobs> <n_machines>
        <job line 1>
        ...
        <job line n_jobs>

        [MACHINE_HOLES]
        <machine_id> <num_holes> <start_1> <duration_1> ... <start_h> <duration_h>

    Example:
        10 10
        4 88 8 68 ...
        ...
        [MACHINE_HOLES]
        0 2 220 35 760 40
        1 1 480 50
        ...
    """

    def __init__(
        self,
        default_alpha: float = 0.0,
        default_crossable: bool = True,
        machine_prefix: str = "M",
        job_prefix: str = "J",
    ) -> None:
        self.default_alpha = default_alpha
        self.default_crossable = default_crossable
        self.machine_prefix = machine_prefix
        self.job_prefix = job_prefix

    def read(self, filepath: str | Path) -> JSPMUInstance:
        path = Path(filepath)
        if not path.exists():
            raise FileNotFoundError(f"File not found: {path}")

        lines = self._clean_lines(path)

        if not lines:
            raise ValueError("Input file is empty.")

        n_jobs, n_machines = self._parse_header(lines[0])

        if len(lines) < 1 + n_jobs:
            raise ValueError(
                f"Expected {n_jobs} job lines after header, found only {len(lines) - 1}."
            )

        jobs = self._parse_jobs(lines[1 : 1 + n_jobs], n_jobs, n_machines)

        # Initialize all machines with empty hole lists
        holes: Dict[str, List[Hole]] = {
            f"{self.machine_prefix}{m}": [] for m in range(n_machines)
        }

        # Optional [MACHINE_HOLES] section
        remainder = lines[1 + n_jobs :]
        if remainder:
            if remainder[0] != "[MACHINE_HOLES]":
                raise ValueError(
                    f"Unexpected content after job lines: {remainder[0]!r}. "
                    "Expected '[MACHINE_HOLES]' or end of file."
                )
            holes = self._parse_holes(remainder[1:], n_machines, holes)

        return JSPMUInstance(
            n_jobs=n_jobs,
            n_machines=n_machines,
            jobs=jobs,
            holes=holes,
        )

    def _clean_lines(self, path: Path) -> List[str]:
        raw_lines = path.read_text(encoding="utf-8").splitlines()
        cleaned: List[str] = []

        for line in raw_lines:
            line = line.strip()
            if not line:
                continue
            # Ignore comment lines
            if line.startswith("#") or line.startswith("//"):
                continue
            cleaned.append(line)

        return cleaned

    def _parse_header(self, line: str) -> tuple[int, int]:
        parts = line.split()
        if len(parts) != 2:
            raise ValueError(
                f"Header must contain exactly two integers: n_jobs n_machines. Got: {line!r}"
            )
        n_jobs, n_machines = map(int, parts)
        if n_jobs <= 0 or n_machines <= 0:
            raise ValueError(
                f"n_jobs and n_machines must be positive. Got: {n_jobs}, {n_machines}"
            )
        return n_jobs, n_machines

    def _parse_jobs(
        self,
        job_lines: List[str],
        n_jobs: int,
        n_machines: int,
    ) -> List[Job]:
        jobs: List[Job] = []

        for job_idx, line in enumerate(job_lines):
            parts = line.split()
            expected = 2 * n_machines
            if len(parts) != expected:
                raise ValueError(
                    f"Job line {job_idx + 1} has {len(parts)} entries; expected {expected}."
                )

            values = list(map(int, parts))
            operations: List[Operation] = []

            for op_idx in range(n_machines):
                machine_id = values[2 * op_idx]
                duration = values[2 * op_idx + 1]

                if not (0 <= machine_id < n_machines):
                    raise ValueError(
                        f"Invalid machine id {machine_id} in job {job_idx + 1}, "
                        f"operation {op_idx + 1}."
                    )

                operations.append(
                    Operation(
                        machine=f"{self.machine_prefix}{machine_id}",
                        duration=float(duration),
                        alpha=self.default_alpha,
                        crossable=self.default_crossable,
                        name=f"O{job_idx + 1}{op_idx + 1}",
                    )
                )

            jobs.append(
                Job(
                    job_id=f"{self.job_prefix}{job_idx + 1}",
                    operations=operations,
                )
            )

        if len(jobs) != n_jobs:
            raise ValueError(f"Parsed {len(jobs)} jobs, expected {n_jobs}.")

        return jobs

    def _parse_holes(
        self,
        hole_lines: List[str],
        n_machines: int,
        holes: Dict[str, List[Hole]],
    ) -> Dict[str, List[Hole]]:
        seen_machines = set()

        for line in hole_lines:
            parts = line.split()
            if len(parts) < 2:
                raise ValueError(f"Invalid machine hole line: {line!r}")

            machine_id = int(parts[0])
            n_holes = int(parts[1])

            if not (0 <= machine_id < n_machines):
                raise ValueError(f"Invalid machine id in hole section: {machine_id}")

            expected = 2 + 2 * n_holes
            if len(parts) != expected:
                raise ValueError(
                    f"Hole line for machine {machine_id} has {len(parts)} entries; "
                    f"expected {expected}."
                )

            machine_name = f"{self.machine_prefix}{machine_id}"
            if machine_name in seen_machines:
                raise ValueError(f"Duplicate hole definition for machine {machine_name}.")
            seen_machines.add(machine_name)

            machine_holes: List[Hole] = []
            for k in range(n_holes):
                start = float(parts[2 + 2 * k])
                duration = float(parts[2 + 2 * k + 1])

                if start < 0:
                    raise ValueError(
                        f"Fixed-hole reader received negative start time for {machine_name}: {start}"
                    )
                if duration <= 0:
                    raise ValueError(
                        f"Hole duration must be positive for {machine_name}. Got {duration}."
                    )

                machine_holes.append(Hole(start=start, end=start + duration))

            # Keep holes sorted by start time
            machine_holes.sort(key=lambda h: (h.start, h.end))
            holes[machine_name] = machine_holes

        return holes

# Import instances

In [9]:
import zipfile
import os

In [10]:
zip_path = "generated_jspmu_instances_litarature.zip"
extract_dir = "generated_jspmu_instances_litarature"

# List to store file names inside the zip
file_names = []

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    file_names = zip_ref.namelist()  # names of all files/folders inside the zip
    zip_ref.extractall(extract_dir)  # unzip everything

print("Files inside zip:")
print(file_names)

Files inside zip:
['abz5_mu_mtbf300_durlognorm2_03.jspmu', 'abz6_mu_mtbf300_durlognorm2_03.jspmu', 'ft06_mu_mtbf300_durlognorm2_03.jspmu', 'ft10_mu_mtbf300_durlognorm2_03.jspmu', 'la01_mu_mtbf300_durlognorm2_03.jspmu', 'la02_mu_mtbf300_durlognorm2_03.jspmu', 'la03_mu_mtbf300_durlognorm2_03.jspmu', 'la04_mu_mtbf300_durlognorm2_03.jspmu', 'la05_mu_mtbf300_durlognorm2_03.jspmu', 'la16_mu_mtbf300_durlognorm2_03.jspmu', 'la17_mu_mtbf300_durlognorm2_03.jspmu', 'la18_mu_mtbf300_durlognorm2_03.jspmu', 'la19_mu_mtbf300_durlognorm2_03.jspmu', 'la20_mu_mtbf300_durlognorm2_03.jspmu', 'orb01_mu_mtbf300_durlognorm2_03.jspmu', 'orb02_mu_mtbf300_durlognorm2_03.jspmu', 'orb03_mu_mtbf300_durlognorm2_03.jspmu', 'orb04_mu_mtbf300_durlognorm2_03.jspmu', 'orb05_mu_mtbf300_durlognorm2_03.jspmu', 'orb06_mu_mtbf300_durlognorm2_03.jspmu', 'orb07_mu_mtbf300_durlognorm2_03.jspmu', 'orb08_mu_mtbf300_durlognorm2_03.jspmu', 'orb09_mu_mtbf300_durlognorm2_03.jspmu', 'orb10_mu_mtbf300_durlognorm2_03.jspmu']


In [11]:
data = []
problematics = []

for instance_name in tqdm(file_names):
  try:
    file_name = f'{extract_dir}/{instance_name}'
    reader = JSPMUReader(default_alpha=0.0, default_crossable=True)
    jspmu_instance = reader.read(file_name)
    id_instance = instance_name.split('_')[0]
    row = {'id':id_instance}

    jobs = jspmu_instance.jobs
    holes = jspmu_instance.holes

    scheduler = FlowScheduler(
          jobs=jobs,
          holes_by_machine=holes,
          rules=["eft", "spt","lpt", "mwr", "least_blocking", "most_operations_remaining","fcfs"],
          rollout_events=0,
          rollout_base_rule="fcfs",
      )

    result = scheduler.run()
    best_makespan = result['makespan']

    row['H-Preemptive'] = int(best_makespan)
    data.append(row)
  except:
    problematics.append(instance_name)
    id_instance = instance_name.split('_')[0]
    row = {'id':id_instance}
    row['H-Preemptive'] = -1
    data.append(row)

100%|██████████| 24/24 [00:00<00:00, 116.22it/s]


In [15]:
df = pd.DataFrame(data)

In [16]:
df

,id,H-Preemptive
0,abz5,1359
1,abz6,1103
2,ft06,-1
3,ft10,1035
4,la01,796
5,la02,830
6,la03,672
7,la04,712
8,la05,614
9,la16,1162


In [17]:
df.to_csv('heuristic_Preemptive_results.csv',index=False)